# SFT training — Alpaca

Fine-tune the pretrained 67M base checkpoint on `tatsu-lab/alpaca`, tokenized with our
own 32k BPE (see `cs336_basics/tokenize_sft.py`).

The tokenized data lives on the **`cs336-sft-data`** Modal volume and is streamed into
memory here (it's ~12 MiB, but per the repo rule nothing big is written to local disk).

**On-disk layout** (`alpaca_vocab32000/`):
- `input_ids.uint16` — flat concatenation of every example's token ids
- `offsets.npy` (int64, `[N+1]`) — example `i` = `input_ids[offsets[i]:offsets[i+1]]`
- `prompt_lens.npy` (int32, `[N]`) — # prompt tokens before the response
- `manifest.json` — metadata

**Loss-mask rule:** for example `i`, train (mask = 1) on positions `>= prompt_lens[i]`
(the response + trailing `<|endoftext|>`); the prompt is conditioning only (mask = 0).

In [1]:
import io, json, sys, pathlib
import numpy as np
import modal

# Resolve repo paths whether the kernel started in repo root or in cs336_basics/.
_cwd = pathlib.Path.cwd()
BASE = _cwd if (_cwd / 'tokenizer_artifacts').exists() else _cwd / 'cs336_basics'
sys.path.insert(0, str(BASE))
from tokenizer import Tokenizer

tokenizer = Tokenizer.from_files(
    BASE / 'tokenizer_artifacts' / 'vocab.json',
    BASE / 'tokenizer_artifacts' / 'merges.json',
    special_tokens=['<|endoftext|>'],
)
EOT_ID = tokenizer.encode('<|endoftext|>')[0]

# Stream a file from the Modal volume into an in-memory buffer (no local file written).
vol = modal.Volume.from_name('cs336-sft-data')
def stream(path):
    buf = io.BytesIO()
    for chunk in vol.read_file(path):
        buf.write(chunk)
    buf.seek(0)
    return buf

print('eot id:', EOT_ID)

eot id: 256


In [3]:
PREFIX = 'alpaca_vocab32000'

input_ids   = np.frombuffer(stream(f'{PREFIX}/input_ids.uint16').read(), dtype=np.uint16)
offsets     = np.load(stream(f'{PREFIX}/offsets.npy'))
prompt_lens = np.load(stream(f'{PREFIX}/prompt_lens.npy'))
manifest    = json.load(stream(f'{PREFIX}/manifest.json'))

print('input_ids  ', input_ids.shape, input_ids.dtype)
print('offsets    ', offsets.shape, offsets.dtype)
print('prompt_lens', prompt_lens.shape, prompt_lens.dtype)
print()
print(json.dumps(manifest, indent=2))

input_ids   (6243135,) uint16
offsets     (52003,) int64
prompt_lens (52002,) int32

{
  "dataset": "tatsu-lab/alpaca",
  "split": "train",
  "tokenizer": "cs336 BPE vocab_size=32000 (owt_prefix_100000_vocab_32000)",
  "special_tokens": {
    "<|endoftext|>": 256
  },
  "template": "alpaca (plain text, response after '### Response:\\n', <|endoftext|> appended)",
  "num_examples": 52002,
  "total_tokens": 6243135,
  "total_response_tokens": 3154705,
  "seq_len": {
    "mean": 120.05567093573325,
    "p50": 105,
    "p95": 226,
    "max": 1566
  },
  "files": {
    "input_ids": "input_ids.uint16 (np.uint16, flat)",
    "offsets": "offsets.npy (int64, [N+1])",
    "prompt_lens": "prompt_lens.npy (int32, [N])"
  },
  "loss_mask_rule": "mask=1 for positions >= prompt_lens[i] (response + <|endoftext|>)"
}


In [4]:
def get_example(i):
    """Return (ids, prompt_ids, response_ids, prompt_len) for example i."""
    ids = input_ids[offsets[i]:offsets[i + 1]]
    plen = int(prompt_lens[i])
    return ids, ids[:plen], ids[plen:], plen
def show(i):
    ids, p_ids, r_ids, plen = get_example(i)
    print(f'==================== example {i} ====================')
    print(f'total tokens: {len(ids)}   prompt: {plen}   response: {len(r_ids)}')
    print('\n----- PROMPT (mask = 0, conditioning) -----')
    print(tokenizer.decode(p_ids.tolist()))
    print('----- RESPONSE (mask = 1, trained on) -----')
    print(repr(tokenizer.decode(r_ids.tolist())))  # repr to make the trailing <|endoftext|> visible

# A no-input example, a with-input example, and a long one.
for i in [0, 1, 5]:
    show(i)
    print()

==================== example 0 ====================
total tokens: 90   prompt: 41   response: 49

----- PROMPT (mask = 0, conditioning) -----
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Give three tips for staying healthy.

### Response:

----- RESPONSE (mask = 1, trained on) -----
'1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.<|endoftext|>'

==================== example 1 ====================
total tokens: 54   prompt: 41   response: 13

----- PROMPT (mask = 0, conditioning) -----
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What are the three primary colors?

### Response:

----- RESPONSE (mask = 1, trained on) -----
'The three primary colors are red, blue, and yellow.<|e

In [ ]:
# Raw token view of one example, so you can see the boundary in id-space.
ids, p_ids, r_ids, plen = get_example(0)
print('prompt_len =', plen)
print('ids       :', ids.tolist())
print('last token:', int(ids[-1]), '== EOT?', int(ids[-1]) == EOT_ID)

## Your turn: data loader + loss masking

You have everything you need:
- `input_ids` (flat uint16), `offsets` (`[N+1]`), `prompt_lens` (`[N]`)
- `EOT_ID` for padding / stop
- `manifest['seq_len']` — mean 120, p95 226, **max 1566** (a few exceed the 512 context)

Things to decide as you build it:
- **Batching variable-length sequences:** pad to the batch max (or to 512) with `EOT_ID`
  (or a pad id), or pack multiple short examples into one 512 window.
- **Inputs vs targets:** standard next-token shift — `x = seq[:-1]`, `y = seq[1:]`.
- **Loss mask:** 1 on positions where the *target* is a response token (index `>= prompt_len`),
  0 on prompt targets and on any padding. Remember the shift when you line the mask up with `y`.
- **Truncation:** decide what to do with sequences longer than `context_length` (512) —
  truncate, or drop ones whose response would be cut.

`np.frombuffer` returns a read-only array; `np.array(input_ids)` if you need to write.

In [ ]:
# TODO(you): SFT data loader
#   def sft_batch(batch_size, context_length, device): -> x, y, loss_mask
# returning input ids, shifted targets, and the response-only loss mask.


In [14]:
i = 2
ids = input_ids[offsets[i]:offsets[i + 1]]
plen = int(prompt_lens[i])
a,b,c,d=ids, ids[:plen], ids[plen:], plen
print("ALL", tokenizer.decode(a.tolist()))
print("MASKED", tokenizer.decode(b.tolist()))
print("PREDICT", tokenizer.decode(c.tolist()))

ALL Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Describe the structure of an atom.

### Response:
An atom is made up of a nucleus, which contains protons and neutrons, surrounded by electrons that travel in orbits around the nucleus. The protons and neutrons have a positive charge, while the electrons have a negative charge, resulting in an overall neutral atom. The number of each particle determines the atomic number and the type of atom.<|endoftext|>
MASKED Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Describe the structure of an atom.

### Response:

PREDICT An atom is made up of a nucleus, which contains protons and neutrons, surrounded by electrons that travel in orbits around the nucleus. The protons and neutrons have a positive charge, while the electrons have a negative charge, resulting in an overall neutral atom. The nu

In [16]:
input_ids

array([18489,   321,   282, ...,  1508,    46,   256],
      shape=(6243135,), dtype=uint16)

In [ ]:
def data_loading_sft(dataset, offsets, batch_size, context_length, device='cpu', dtype=torch.int):
    x_starts = np.random.randint(0, dataset.shape[0]-context_length, batch_size).reshape(-1, 1)
    offsets = np.arange(context_length).reshape(1, -1)
    x_index = offsets + x_starts
    y_index = x_index + 1
    return torch.tensor(dataset[x_index], device=device, dtype=dtype), torch.tensor(dataset[y_index], device=device    , dtype=dtype)

In [50]:
batch_size = 20
dataset = input_ids
context_length=512

In [57]:

starts = np.random.randint(0, offsets.shape[0] - 2, batch_size)
x = np.zeros((batch_size, context_length))
loss_mask = np.zeros((batch_size, context_length))
y = np.zeros((batch_size, context_length))
for i, start in enumerate(starts):
    input_start = offsets[start]
    input_end = offsets[start+1]
    if input_end - input_start > context_length: continue
    _x = input_ids[input_start:input_end]
    _y = input_ids[input_start+1:input_end+1]
    x[i, 0:_x.shape[0]] = _x
    y[i, 0:_x.shape[0]] = _y
    loss_mask[i, int(prompt_lens[start])-1:_x.shape[0]-1] = 1

    
    
    
    

93 139
26 93
125 176
84 128
29 77
53 132
95 141
11 57
120 170
61 110
35 84
10 55
42 86
147 203
27 73
128 201
130 176
94 158
14 59
9 85


In [54]:
x[0],y[0],loss_mask[0]

(array([1.8489e+04, 3.2100e+02, 2.8200e+02, 1.2484e+04, 3.1800e+02,
        7.9600e+03, 2.5800e+02, 4.9690e+03, 4.4000e+01, 1.8000e+04,
        3.5200e+02, 2.8200e+02, 5.8270e+03, 3.1800e+02, 3.8290e+03,
        2.2320e+03, 5.1800e+03, 4.6000e+01, 2.9055e+04, 2.5800e+02,
        2.7310e+03, 3.1800e+02, 1.8755e+04, 1.2190e+03, 5.1060e+03,
        2.6200e+02, 2.5830e+03, 4.6000e+01, 1.0000e+01, 1.0000e+01,
        2.6516e+04, 3.5000e+01, 2.2040e+03, 2.8300e+03, 5.8000e+01,
        1.0000e+01, 1.9906e+04, 2.6200e+02, 1.0220e+04, 2.8800e+02,
        2.6200e+02, 1.7970e+03, 3.7560e+03, 1.9510e+03, 4.2700e+02,
        2.6200e+02, 4.8200e+03, 4.6000e+01, 1.0000e+01, 1.0000e+01,
        2.6516e+04, 3.5000e+01, 5.9000e+02, 2.4960e+03, 5.8000e+01,
        1.0000e+01, 8.6260e+03, 7.6330e+03, 5.8000e+01, 4.4100e+02,
        1.4170e+03, 2.1462e+04, 4.0000e+02, 1.0000e+01, 8.3000e+01,
        3.0000e+02, 5.8900e+02, 5.8000e+01, 1.5000e+03, 3.5840e+03,
        5.4500e+02, 7.2080e+03, 3.1800e+02, 6.23

In [55]:
tokenizer.decode(x[0].tolist())

'Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nFind the index of the given substring from the sentence.\n\n### Input:\nSubstring: “application”\nSentence: She felt so excited that her application got approval.\n\n### Response:\n15<|endoftext|>\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00

In [33]:
ends = starts + 1

In [34]:
starts, ends

(array([[39465],
        [19028],
        [18914]]),
 array([[39466],
        [19029],
        [18915]]))

In [40]:
tokenizer.decode()

'Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nCreate a mnemonic device to help a student remember the names of four planets\n\n### Input:\nPlanets: Mercury, Venus, Earth, Mars\n\n### Response:\nM - Mercury\nV - Venus\nE - Earth\nM - Mars<|endoftext|>'

In [24]:
input_ids[0:90].tolist())

'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.<|endoftext|>'